# Module 01 · Unit 01
# Propositions and Logical Connectives

| | |
|---|---|
| **Estimated time** | 50–60 min |
| **Exercises** | Download PDF from the course repository |
| **Security thread** | Access Control & Policy Logic \[AC\] |
| **Refresher** | Module 00 · Unit 01 — Proof Techniques |

---

## Learning Objectives

By the end of this notebook you will be able to:

1. Identify atomic and compound propositions
2. Apply all five logical connectives: $\neg$, $\wedge$, $\vee$, $\rightarrow$, $\leftrightarrow$
3. Evaluate compound propositions using operator precedence
4. Explain why the conditional $\rightarrow$ behaves differently from everyday "if...then"
5. Recognise each connective in real access control policy language
6. Use Python to generate truth tables and visualize connective behaviour


## 🔣 Symbol Primer

This notebook introduces three new connective symbols. The first two ($\neg$ and
$\wedge$) were glossed in Module 00 — they are repeated here for quick reference.

| Symbol | Name | Read it as | True when... |
|---|---|---|---|
| $\neg P$ | Negation | "not $P$" | $P$ is false |
| $P \wedge Q$ | Conjunction | "$P$ and $Q$" | **both** $P$ and $Q$ are true |
| $P \vee Q$ | Disjunction | "$P$ or $Q$" | **at least one** of $P$, $Q$ is true |
| $P \rightarrow Q$ | Conditional | "if $P$ then $Q$" | $P$ is false, **or** $Q$ is true |
| $P \leftrightarrow Q$ | Biconditional | "$P$ if and only if $Q$" | $P$ and $Q$ have the **same** truth value |
| $P \equiv Q$ | Logical equivalence | "$P$ is equivalent to $Q$" | $P$ and $Q$ are true in exactly the same cases |

> **New in this notebook:** $\vee$, $\rightarrow$, $\leftrightarrow$, $\equiv$.  
> **Precedence** (tightest first): $\neg$ then $\wedge$ then $\vee$ then $\rightarrow$ then $\leftrightarrow$.  
> Think of it like arithmetic: $\neg$ is exponentiation, $\wedge$ is multiplication, $\vee$ is addition.

---


## 1 · Atomic and Compound Propositions

In Module 00 we defined a **proposition** as a statement with a definite truth value.
Propositions come in two varieties:

An **atomic proposition** has no internal logical structure — it is the smallest unit
of meaning. We assign it a single letter:

$$P : \text{"The user's role is admin."}$$
$$Q : \text{"The current time is within the allowed window."}$$
$$S : \text{"The account is suspended."}$$

A **compound proposition** is built from atomic propositions using **logical
connectives** — the symbols $\neg$, $\wedge$, $\vee$, $\rightarrow$, $\leftrightarrow$.
For example:

$$P \wedge Q \wedge \neg S : \text{"Role is admin, time is in window, and account is not suspended."}$$

This compound proposition is exactly the structure of an attribute-based access control
(ABAC) policy. The five connectives are the vocabulary of every access control system,
every protocol specification, and every formal security property in this course.


## 2 · The Five Connectives

### 2.1 · Negation — $\neg$

$\neg P$ *(not $P$)* flips the truth value of $P$.

| $P$ | $\neg P$ |
|:---:|:---:|
| T | F |
| F | T |

**Security reading:** $\neg\,\text{suspended}$ — "the account is *not* suspended."
A policy that includes $\neg S$ requires that suspended accounts are **explicitly
checked and blocked.** If the check is skipped, $S$ is never evaluated.

---

### 2.2 · Conjunction — $\wedge$

$P \wedge Q$ *(P and Q)* is true only when *both* operands are true. One false input
makes the whole conjunction false.

| $P$ | $Q$ | $P \wedge Q$ |
|:---:|:---:|:---:|
| T | T | T |
| T | F | F |
| F | T | F |
| F | F | F |

**Security reading:** $P \wedge Q \wedge \neg S$ — access requires **all** conditions
simultaneously. Conjunction is the natural language of *restrictive* policy: add another
$\wedge$ clause to tighten the rule.

---

### 2.3 · Disjunction — $\vee$ *(new)*

$P \vee Q$ *(P or Q)* is true when *at least one* operand is true. Both true is also
fine — this is **inclusive or**, not exclusive or.

| $P$ | $Q$ | $P \vee Q$ |
|:---:|:---:|:---:|
| T | T | T |
| T | F | T |
| F | T | T |
| F | F | F |

**Security reading:** $\text{admin} \vee \text{owner}$ — "access if the user is an
admin *or* the resource owner." Disjunction is *permissive*: each additional $\vee$
clause widens the gate. This is where policy over-provisioning typically happens.

---

### 2.4 · Conditional — $\rightarrow$ *(new)*

$P \rightarrow Q$ *(if P then Q)* is the connective that surprises people most.
It is **false only when $P$ is true and $Q$ is false** — that is, when the premise
holds but the promised conclusion does not.

| $P$ | $Q$ | $P \rightarrow Q$ |
|:---:|:---:|:---:|
| T | T | T |
| T | F | **F** |
| F | T | T |
| F | F | T |

The last two rows are the counterintuitive ones. When $P$ is false, the conditional is
true regardless of $Q$ — a false premise makes no promise, so no promise is broken.
This is called **vacuous truth**.

**Security reading:** $\text{privileged\_op} \rightarrow \text{audit\_logged}$ —
"if an operation is privileged, then it must be audit-logged." This is false (a
violation) exactly in the one case you care about: a privileged operation that goes
unlogged.

---

### 2.5 · Biconditional — $\leftrightarrow$ *(new)*

$P \leftrightarrow Q$ *(P if and only if Q)* is true when both sides have the
**same truth value** — both true, or both false.

| $P$ | $Q$ | $P \leftrightarrow Q$ |
|:---:|:---:|:---:|
| T | T | T |
| T | F | F |
| F | T | F |
| F | F | T |

You can read $P \leftrightarrow Q$ as $(P \rightarrow Q) \wedge (Q \rightarrow P)$
— the implication goes both ways.

**Security reading:** $\text{key\_revoked} \leftrightarrow \text{access\_denied}$ —
key revocation and access denial must be kept in perfect sync. If one changes without
the other, the biconditional is violated and a security gap exists.


## 3 · Operator Precedence

Just as multiplication binds tighter than addition in arithmetic, logical connectives
have a binding order. From tightest to loosest:

$$\neg \;>\; \wedge \;>\; \vee \;>\; \rightarrow \;>\; \leftrightarrow$$

So $\neg P \wedge Q \vee R$ parses as $((\neg P) \wedge Q) \vee R$, not as
$\neg(P \wedge (Q \vee R))$.

**When in doubt, use parentheses.** In formal proofs and in code, explicit parentheses
are never wrong and often clarify intent.

### Worked Example — Parsing an Access Policy

Consider the policy fragment:

$$\neg S \wedge P \vee Q \rightarrow \text{allow}$$

Applying precedence step by step:

1. $\neg$ first: $\;(\neg S) \wedge P \vee Q \rightarrow \text{allow}$
2. $\wedge$ next: $\;((\neg S) \wedge P) \vee Q \rightarrow \text{allow}$
3. $\vee$ next: $\;(((\neg S) \wedge P) \vee Q) \rightarrow \text{allow}$
4. $\rightarrow$ last: the full formula

Read: *"If (not suspended AND privileged) OR owner, then allow."*

> **Security implication.** The placement of parentheses changes who gets access.
> $\neg S \wedge (P \vee Q)$ means *"not suspended AND (privileged OR owner)"* —
> suspended accounts are always blocked. Without the parentheses, a resource owner
> who is suspended still gets access via the $\vee Q$ branch. This is a real class
> of access control misconfiguration.


## 4 · Logical Equivalence and De Morgan's Laws

Two propositions are **logically equivalent** ($\equiv$ — *is equivalent to*) when
they have identical truth values for every possible assignment of their variables.
Equivalence means you can substitute one for the other anywhere without changing meaning.

### De Morgan's Laws

The two most important equivalences in both logic and security engineering:

$$\neg (P \wedge Q) \;\equiv\; (\neg P \vee \neg Q)$$

$$\neg (P \vee Q) \;\equiv\; (\neg P \wedge \neg Q)$$

**How to remember them:** negation distributes over a conjunction/disjunction and
*flips the connective* — $\wedge$ becomes $\vee$ and vice versa.

### Why De Morgan's Laws Matter for Access Control

Suppose a policy allows access when $P \wedge Q$ — both conditions must hold.
What is the condition for **denying** access? By De Morgan's first law:

$$\neg(P \wedge Q) \;\equiv\; \neg P \vee \neg Q$$

*"Deny if role is wrong **or** time is out of window."*

This is how policy engines compute denial conditions from allow conditions, and it is
how attackers reason about which single condition to subvert. Breaking *either*
$P$ or $Q$ is sufficient to bypass an AND policy.

We will verify De Morgan's laws formally with truth tables in Unit 02, and apply
them to simplify ABAC policies algebraically in Unit 03.


## 5 · A First Access Control Policy

We now have enough vocabulary to read and write a real access control policy as a
formal proposition. Consider:

> *"Allow access if the user's role is admin, the request is within the allowed time
> window, and the account is not suspended."*

Let:
- $A$ = "role is admin"
- $T$ = "time is within allowed window"
- $S$ = "account is suspended"

The policy in formal logic:

$$\text{allow} \;\equiv\; A \wedge T \wedge \neg S$$

This is an **Attribute-Based Access Control (ABAC)** policy expressed as a
conjunction of three conditions. Every clause tightens the gate:

| Clause | Connective | Security role |
|---|---|---|
| $A$ | $\wedge$ (and) | Role check — only admins proceed |
| $T$ | $\wedge$ (and) | Time-of-day restriction |
| $\neg S$ | $\wedge$ (and) | Suspension check — blocked accounts denied |

### The Short-Circuit Question

Most programming languages evaluate $\wedge$ (logical AND) with **short-circuit
evaluation**: if the first operand is false, the second is never evaluated.

Consider this Python expression:

```python
allow = is_admin(user) and in_time_window(request) and not is_suspended(user)
```

If `is_admin` returns `False`, Python never calls `is_suspended`. In this case
that is harmless — access is already denied. But consider a different ordering:

```python
allow = in_time_window(request) and is_admin(user) and not is_suspended(user)
```

If `in_time_window` returns `False`, `is_suspended` is never called. If the
suspension check also triggers a security audit log, that log entry is now missing.
The **logical result** is the same (access denied), but the **side effects** differ.

This is the shape of a class of security bugs we will analyze formally in Unit 03
when we build the full truth table for this policy. For now: **the order of
conjuncts matters for operational correctness even when it does not change the
logical outcome.**


---

## 🔐 Security Bridge &nbsp;·&nbsp; \[AC\]

Every logical connective introduced in this notebook has a direct counterpart in
access control system design:

| Connective | Policy role | Risk if misused |
|---|---|---|
| $\neg$ (not) | Exclusion clause: block suspended, revoked, banned | Skipped check → unconditional access |
| $\wedge$ (and) | Restrictive rule: ALL conditions must hold | Short-circuit → side-effect bypass |
| $\vee$ (or) | Permissive rule: ANY condition grants access | Over-provisioning → privilege creep |
| $\rightarrow$ (if...then) | Obligation: if privileged, then audit | Vacuous truth → unlogged operations |
| $\leftrightarrow$ (iff) | Synchronization: two states must match | Drift → revocation gap |

**The connective that causes the most real-world bugs is $\rightarrow$.**
Security obligations are usually written as conditionals: *"if sensitive, then
encrypt"*, *"if admin, then MFA required"*, *"if external request, then rate-limit."*
The vacuous truth of the conditional means that if the antecedent is never true —
because the classification system mislabels data, because admin detection fails,
because the external/internal check has a logic error — the obligation is
silently satisfied without the protection ever activating.

We will formalize this into a complete analysis using truth tables in Unit 02.

---


## Python: Visualization & Verification

**1 · Truth Table Generator** — build a reusable function that evaluates any
Boolean formula over all possible inputs and displays the result as a formatted table.

**2 · Connective Comparison** — visualize all five connectives side by side across
every input combination, making their differences immediately visible.

**3 · Precedence in Action** — demonstrate how parenthesis placement changes a
compound formula's truth table, using the access policy example from Section 3.


In [ ]:
import subprocess, sys

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

for pkg in ["numpy", "matplotlib", "sympy", "scipy", "networkx"]:
    install(pkg)

print("All packages installed.")


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import sympy as sp
import networkx as nx
import itertools

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (9, 6), 'font.size': 11,
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'lines.linewidth': 2, 'figure.dpi': 120,
})

TS_BLUE  = '#1e64b4'
TS_AMBER = '#c87814'
TS_GREEN = '#1e8c50'
TS_GRAY  = '#64646e'
TS_RED   = '#b41e1e'
TS_LIGHT = '#f5f7fa'

print('Setup complete.')


### 1 · Truth Table Generator

A **truth table** exhaustively lists the output of a compound proposition for every
possible combination of its input truth values. For $n$ variables there are $2^n$ rows.

The function below takes a list of variable names and a formula expressed as a Python
lambda, generates all $2^n$ input combinations, evaluates the formula for each, and
prints a formatted table. We will use this tool throughout the module.

We start with two familiar formulas, then move to the one that surprises students most:
the conditional $P \rightarrow Q$.


In [ ]:
# ── 1 · Truth Table Generator ─────────────────────────────────────────────────

def truth_table(variables, formula, formula_label):
    """Print a truth table for a Boolean formula.

    Args:
        variables    : list of variable name strings, e.g. ['P', 'Q']
        formula      : callable taking keyword args matching variables
        formula_label: string label for the output column header
    """
    combos = list(itertools.product([True, False], repeat=len(variables)))
    header = variables + [formula_label]
    col_w  = max(len(h) for h in header) + 2

    # Header
    print('  '.join(h.center(col_w) for h in header))
    print('  '.join('-' * col_w for _ in header))

    for combo in combos:
        assignment = dict(zip(variables, combo))
        result     = formula(**assignment)
        vals = [('T' if v else 'F') for v in combo] + [('T' if result else 'F')]
        print('  '.join(v.center(col_w) for v in vals))
    print()

# ── Conjunction and Disjunction ────────────────────────────────────────────────
print("P ∧ Q  (conjunction — AND)")
truth_table(['P', 'Q'], lambda P, Q: P and Q, 'P ∧ Q')

print("P ∨ Q  (disjunction — OR)")
truth_table(['P', 'Q'], lambda P, Q: P or Q, 'P ∨ Q')

# ── The Conditional: the surprising one ───────────────────────────────────────
print("P → Q  (conditional — IF P THEN Q)")
print("Notice: rows 3 and 4 — when P is False, the output is always True.")
print("A false premise makes no promise, so no promise is broken (vacuous truth).\n")
truth_table(['P', 'Q'], lambda P, Q: (not P) or Q, 'P → Q')

# ── Biconditional ─────────────────────────────────────────────────────────────
print("P ↔ Q  (biconditional — P IFF Q)")
truth_table(['P', 'Q'], lambda P, Q: P == Q, 'P ↔ Q')


**What do you see?**

- $P \wedge Q$: only one T in the output — the top row. Everything else is F.
- $P \vee Q$: only one F — the bottom row. Everything else is T.
- $P \rightarrow Q$: **three** T rows and only one F. The conditional is much more
  permissive than it looks in English. It is false only on row 2 (T→F).
- $P \leftrightarrow Q$: T when both match (TT or FF), F when they differ.

**Try this:** Modify the conditional formula to encode the audit obligation from
Section 2.4: $\text{privileged} \rightarrow \text{logged}$. Which row represents
"a privileged operation that was not logged" — the security violation?


### 2 · All Five Connectives Side by Side

Seeing all five connectives in a single visual makes their differences immediate.
The heatmap below shows truth values (green = True, red = False) for every connective
across all four input combinations of two variables $P$ and $Q$.


In [ ]:
# ── 2 · Connective Comparison Heatmap ────────────────────────────────────────

combos     = [(T, T), (T, F), (F, T), (F, F)]
row_labels = ['T, T', 'T, F', 'F, T', 'F, F']
connectives = {
    r'$\neg P$':          [not P          for P, Q in combos],
    r'$P \wedge Q$':      [P and Q        for P, Q in combos],
    r'$P \vee Q$':        [P or Q         for P, Q in combos],
    r'$P \rightarrow Q$': [(not P) or Q   for P, Q in combos],
    r'$P \leftrightarrow Q$': [P == Q     for P, Q in combos],
}

# Build matrix: rows = input combos, cols = connectives
col_labels = list(connectives.keys())
matrix     = np.array([connectives[k] for k in col_labels], dtype=float).T
# matrix shape: (4 combos, 5 connectives)

fig, ax = plt.subplots(figsize=(10, 4))
cmap = ListedColormap([TS_RED, TS_GREEN])
im   = ax.imshow(matrix, cmap=cmap, vmin=0, vmax=1, aspect='auto')

# Annotate cells
for r in range(matrix.shape[0]):
    for c in range(matrix.shape[1]):
        val = 'T' if matrix[r, c] else 'F'
        ax.text(c, r, val, ha='center', va='center',
                fontsize=13, fontweight='bold', color='white')

ax.set_xticks(range(len(col_labels)))
ax.set_xticklabels(col_labels, fontsize=11)
ax.set_yticks(range(len(row_labels)))
ax.set_yticklabels([f'P={r[0]}, Q={r[2]}' for r in row_labels], fontsize=10)
ax.set_title('All Five Connectives — Truth Values for Every Input Combination',
             pad=12)
ax.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False)

# Legend
handles = [mpatches.Patch(color=TS_GREEN, label='True'),
           mpatches.Patch(color=TS_RED,   label='False')]
ax.legend(handles=handles, loc='lower right',
          bbox_to_anchor=(1.18, 0), fontsize=9)

plt.tight_layout()
plt.show()

# Key observations printed below the plot
print("Key patterns:")
print("  ¬P      — depends only on P, ignores Q entirely")
print("  P ∧ Q   — exactly ONE True row (both true)")
print("  P ∨ Q   — exactly ONE False row (both false)")
print("  P → Q   — exactly ONE False row (P=T, Q=F) — the broken promise")
print("  P ↔ Q   — True on the diagonal (TT and FF) — same truth value")


**What do you see?**

The heatmap makes the structure of each connective visible at a glance:

- **$\neg P$** is a vertical stripe — it only cares about $P$, not $Q$.
- **$\wedge$** has one green cell: the most restrictive connective.
- **$\vee$** has three green cells: the most permissive of the binary connectives.
- **$\rightarrow$** has three green cells like $\vee$ — and in fact
  $P \rightarrow Q \equiv \neg P \vee Q$. We will prove this identity
  formally with a truth table in Unit 02.
- **$\leftrightarrow$** is green on the "matching" rows (TT, FF) and red on
  the "mismatched" rows (TF, FT) — it is testing for equality.

Notice how $\rightarrow$ and $\vee$ have the same number of green cells but in
different positions. This is why they are not equivalent — the one red cell is
in a different row.


### 3 · Precedence in Action — When Parentheses Change Everything

We saw in Section 3 that the placement of parentheses in an access policy can
change who gets access. Let's make this concrete with truth tables.

We compare two policies over three variables:
- $A$ = role is admin
- $Q$ = user is resource owner
- $S$ = account is suspended

**Policy X** (no explicit parentheses, precedence applies):
$$\neg S \wedge A \vee Q$$
which parses as $(\neg S \wedge A) \vee Q$

**Policy Y** (parentheses make intent explicit):
$$\neg S \wedge (A \vee Q)$$

These look similar but grant access to different sets of users. The difference is
exactly in the suspended owner case: does suspension override ownership?


In [ ]:
# ── 3 · Precedence — Two Policies Compared ────────────────────────────────────

variables = ['A (admin)', 'Q (owner)', 'S (suspended)']
combos_3  = list(itertools.product([True, False], repeat=3))

policy_x = lambda A, Q, S: ((not S) and A) or Q   # (¬S ∧ A) ∨ Q
policy_y = lambda A, Q, S: (not S) and (A or Q)    # ¬S ∧ (A ∨ Q)

# Compute results
results = []
for A, Q, S in combos_3:
    x = policy_x(A=A, Q=Q, S=S)
    y = policy_y(A=A, Q=Q, S=S)
    results.append((A, Q, S, x, y, x != y))

# ── Visual table ───────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(11, 5))
ax.axis('off')

col_headers = ['A\n(admin)', 'Q\n(owner)', 'S\n(suspended)',
               'Policy X\n(¬S∧A)∨Q', 'Policy Y\n¬S∧(A∨Q)', 'Differ?']

table_data = []
cell_colors = []
for A, Q, S, x, y, diff in results:
    row = ['T' if v else 'F' for v in [A, Q, S, x, y]] + ['⚠ YES' if diff else '—']
    table_data.append(row)
    row_colors = ['#f8f9fa'] * 3  # neutral for inputs
    row_colors.append(TS_GREEN if x else TS_RED)
    row_colors.append(TS_GREEN if y else TS_RED)
    row_colors.append('#fff3cd' if diff else '#f8f9fa')
    cell_colors.append(row_colors)

tbl = ax.table(
    cellText=table_data,
    colLabels=col_headers,
    cellColours=cell_colors,
    cellLoc='center',
    loc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(10)
tbl.scale(1.2, 1.6)

# Style header row
for j in range(len(col_headers)):
    tbl[(0, j)].set_facecolor(TS_BLUE)
    tbl[(0, j)].set_text_props(color='white', fontweight='bold')

ax.set_title(
    'Policy X: $(\\neg S \\wedge A) \\vee Q$  vs  '
    'Policy Y: $\\neg S \\wedge (A \\vee Q)$\n'
    'Same variables, different parentheses — different access decisions',
    pad=14, fontsize=11
)
plt.tight_layout()
plt.show()

# Summarise differences
diff_cases = [(A, Q, S) for A, Q, S, x, y, diff in results if diff]
print(f"Rows where policies differ: {len(diff_cases)}")
for A, Q, S in diff_cases:
    print(f"  A={'T' if A else 'F'}, Q={'T' if Q else 'F'}, S={'T' if S else 'F'}"
          f"  →  X grants: {policy_x(A=A,Q=Q,S=S)}, Y grants: {policy_y(A=A,Q=Q,S=S)}")
print()
print("Interpretation: Policy X allows a SUSPENDED OWNER to access the resource.")
print("Policy Y blocks ALL suspended users regardless of role or ownership.")
print("Which policy was intended? Parentheses make the answer unambiguous.")


**What do you see?**

There are exactly **two rows** where the policies differ — both involve a suspended
owner ($Q = T$, $S = T$):

- **Policy X** — $(\neg S \wedge A) \vee Q$ — grants access to suspended owners.
  The $\vee Q$ branch fires regardless of suspension status. A suspended user who
  happens to be a resource owner walks straight through.

- **Policy Y** — $\neg S \wedge (A \vee Q)$ — blocks all suspended accounts.
  The $\neg S$ check gates the entire policy; no other clause can bypass it.

**The precedence mistake:** Without explicit parentheses, $\wedge$ binds tighter than
$\vee$, so $\neg S \wedge A \vee Q$ silently becomes Policy X. A developer who
*intended* Policy Y but wrote the expression without parentheses has introduced a
privilege escalation vulnerability for suspended resource owners.

This is precisely the shape of real access control bugs. The fix is always the
same: state your intent with parentheses, then verify with a truth table.


In [ ]:
# ── Extension Challenge ───────────────────────────────────────────────────────
#
# 1. VACUOUS TRUTH
#    The conditional P → Q is True whenever P is False.
#    Write a policy lambda for: "If the request is external, then rate-limit applies."
#    Variables: E = "external request", R = "rate limit applied"
#    Generate the truth table. Which row represents the security obligation being met
#    vacuously (i.e., the protection is never needed but also never applied)?
#    Is that a problem? Why or why not?
#
# 2. DE MORGAN IN CODE
#    Use the truth_table function to verify De Morgan's first law:
#      ¬(P ∧ Q)  ≡  (¬P ∨ ¬Q)
#    Generate both truth tables and print them side by side.
#    Add an assertion that confirms they are identical for all input combinations.
#
# 3. THREE-CLAUSE POLICY
#    Extend the precedence comparison to a four-variable policy:
#      A = admin, O = owner, T = time-in-window, S = suspended
#    Write two versions of: "admin or owner, during the window, and not suspended"
#    with and without explicit parentheses. Do they differ? Under which conditions?

# Your work here:


---

## Summary

| Connective | Symbol | False when... | Security role |
|---|---|---|---|
| Negation | $\neg P$ | $P$ is true | Exclusion: not suspended, not revoked |
| Conjunction | $P \wedge Q$ | Either operand is false | Restrictive AND — tightens the gate |
| Disjunction | $P \vee Q$ | Both operands are false | Permissive OR — widens the gate |
| Conditional | $P \rightarrow Q$ | $P$ true and $Q$ false | Obligation: if privileged, then audit |
| Biconditional | $P \leftrightarrow Q$ | $P$ and $Q$ differ | Sync: revocation ↔ access denial |

**Key takeaways:**

- $\rightarrow$ is the connective that bites hardest in security — three True rows
  means obligations are silently satisfied in most cases, masking the one violation.
- Parentheses are not optional decoration. In compound policies, they determine
  which users get access. Always write them explicitly.
- De Morgan's laws connect AND-based allow policies to OR-based deny conditions —
  a tool we will use heavily in Unit 03.

---

## Up Next

**Module 01 · Unit 02 — Truth Tables**

We build the full truth table methodology: tautologies, contradictions, satisfiability,
and logical equivalence. We prove De Morgan's laws formally and use truth tables to
expose a short-circuit evaluation bug in the ABAC policy from Section 5.

$\rightarrow$ `modules/module-01/unit-02-truth-tables.ipynb`
